In [1]:
import os
import numpy as np
import subprocess
device = 0

In [2]:
ckp, dp = './metadata/DINOv2_checkpoint/', '/mnt/extdisk/azamanos_data/particles_h5/'

eval_tasks = ('Symmetry_ue_train', 'ProteinClass_ue_train', 'MolecularWeight_kDa_ue_train',
              'ImagePixelSize_ue_train', 'EMD_Resolution_ue_train',  'DefocusV_ue_train',
              
              'Symmetry_ue_train_on_eval', 'ProteinClass_ue_train_on_eval', 'MolecularWeight_kDa_ue_train_on_eval',
              'ImagePixelSize_ue_train_on_eval', 'EMD_Resolution_ue_train_on_eval',  'DefocusV_ue_train_on_eval')

In [3]:
tcmd = ''
for p in eval_tasks:
    if f'Linear_{p}' in os.listdir(ckp):
        if 'results-linear.csv' in os.listdir(f'{ckp}/Linear_{p}/'):
            continue
    print(p)
    tcmd += f'CUDA_VISIBLE_DEVICES={device} PYTHONPATH=${{PWD}} python dinov3/eval/linear.py \
            model.config_file={ckp}config.yaml \
            model.pretrained_weights={ckp}teacher_checkpoint.pth \
            output_dir={ckp}Linear_{p}/ \
            train.dataset=MultiH5:root={dp}:h5_key=images:parquet_annotations=./metadata/parquet_files/train/{p}.parquet \
            train.val_dataset=MultiH5:root={dp}:h5_key=images:parquet_annotations=./metadata/parquet_files/eval/{p}.parquet && '

In [4]:
CONDA_SH = "/home/azamanos/miniconda3/etc/profile.d/conda.sh"

cmd = f"""
source {CONDA_SH}
conda activate dinov3
export MKL_THREADING_LAYER=GNU
{tcmd[:-3]}
"""

subprocess.run(['/bin/bash', '-lc', cmd])